# **CLDT - Thread - Panduan Implementasi 10 Hari Pertama**
## **Fondasi Portable dan Gerbang Fisik UDP Satu Endpoint**

Panduan kerja untuk file, kontrak, build, dan hardware yang memang berada pada jalur Days 1-10 di commit `0dee73247d4e87a3afec8fb90f37439cbb96523b`. Snippet kode mempertahankan TODO asli agar implementasinya tetap dikerjakan di source repository. Cakupan berhenti pada cold boot dan UDP berulang melalui S3, C6 RCP, serta satu endpoint.

**Scope & Tags:** `Lembar Kerja Implementasi` | `Urutan Kerja 1-10` | `Upstream First` | `Satu Endpoint`

---

### **Scope & Boundary Specifications**

| Domain | Specification & Minimum Requirements |
| :--- | :--- |
| **Hasil Minimum** | RCP dan border router upstream menyala konsisten, satu endpoint bergabung, lalu UDP bertahan selama jendela uji yang dibekukan sebelum pengujian. |
| **Kode Project** | Kontrak wire/auth dibekukan pada header yang sudah ada. Implementasi portable pertama menutup TODO CRC - 32C dan test fixed - vector yang sudah terdaftar oleh CMake. |
| **Batas Kerja** | Endpoint kedua, protocol CLDT, EDF, recorder, model, fidelity gate, aktuasi, SMP, power, dan eksperimen reportable tetap di luar gerbang ini. |


## **Peta Kerja 1-10**
### **Dari Contract Freeze ke Gerbang UDP Satu Endpoint**

Diagram ini menghubungkan pekerjaan kode dan pekerjaan meja hardware yang berjalan paralel. Panah menuju gerbang hanya menunjukkan dependency; ia tidak menyatakan bahwa langkahnya sudah lulus.

```mermaid
flowchart LR
    subgraph freeze["Days 1-3  -  Contract freeze"]
        bom["hardware/BOM.md<br/>board, pin map, anggaran"]
        wire["cldt_types.h + cldt_protocol.h<br/>wire layout dan semantics"]
        auth["cldt_auth.h + SECURITY.md<br/>auth coverage dan key lifecycle"]
        toolchain["ESP-IDF checkout<br/>commit, sdkconfig, binary digest"]
    end

    subgraph portable["Portable foundation"]
        build["CMakeLists.txt<br/>common/CMakeLists.txt<br/>tests/CMakeLists.txt"]
        crc["cldt_crc32c.h + cldt_crc32c.c<br/>CRC-32C/Castagnoli contract"]
        vector["test_crc32c.c<br/>known-answer + incremental vectors"]
        ci["scaffold-validation.yml<br/>build, CTest, manifest syntax"]
        build --> crc --> vector --> ci
    end

    subgraph physical["Upstream physical path"]
        rcp["ESP-IDF ot_rcp<br/>C6 RCP build + flash"]
        br["ESP-IDF ot_br<br/>S3 host + UART Spinel"]
        cli["ESP-IDF ot_cli<br/>C6 endpoint A attach"]
        udp["Thread UDP<br/>bidirectional datagram test"]
        rcp --> br --> cli --> udp
    end

    freeze --> portable
    bom --> rcp
    toolchain --> rcp
    toolchain --> br
    toolchain --> cli
    portable --> gate{"Day-10 gate"}
    udp --> gate
    gate -->|"cold boot repeatable + sustained UDP"| week2["Masuk Week 2<br/>local accounting, lalu endpoint B"]
    gate -->|"belum repeatable"| cut["Pertahankan satu endpoint<br/>potong model dan control depth"]

    classDef contract fill:#312e81,stroke:#a78bfa,color:#f5f3ff;
    classDef code fill:#0c4a6e,stroke:#38bdf8,color:#e0f2fe;
    classDef hardware fill:#134e4a,stroke:#2dd4bf,color:#ccfbf1;
    classDef decision fill:#713f12,stroke:#fbbf24,color:#fef3c7;
    class wire,auth contract;
    class build,crc,vector,ci code;
    class bom,toolchain,rcp,br,cli,udp,week2 hardware;
    class gate,cut decision;
```


## **Workspace Header - Salinan Persis Repo**
### **Skeletal Contract yang Diselesaikan Sebelum Source**

Empat header berikut adalah salinan persis file repository yang menjadi titik kerja Days 1-3. Deklarasi yang sudah ada dipertahankan; bagian yang belum normatif diselesaikan di header dan dokumentasi terkait sebelum badan source mengandalkan asumsi tersebut. Tidak ada interface alternatif yang dibuat di notebook.

### **`common/include/cldt/cldt_status.h`**

Pekerjaan pada header ini: cocokkan setiap failure boundary yang sudah direncanakan dengan satu nilai `cldt_status_t`. Nilai baru hanya layak ditambah bila kegagalannya memang berbeda secara semantik dan akan direkam sebagai evidence.

~~~c
#ifndef CLDT_STATUS_H
#define CLDT_STATUS_H

#ifdef __cplusplus
extern "C" {
#endif

/*
 * Status values are stable public contracts. A caller records the exact status
 * at a trust, parsing, queue, or transport boundary; it must not convert an
 * error into success merely to keep a run moving. Platform adapters may map an
 * ESP-IDF or broker error to CLDT_ERR_IO, but semantic failures below remain
 * distinct so evidence can explain why a message or policy was rejected.
 */
typedef enum {
    CLDT_OK = 0,
    /* Caller or callee pointer/range/precondition was invalid. */
    CLDT_ERR_INVALID_ARGUMENT = -1,
    /* Bounded caller-owned storage, pool, queue, or encoder output was full. */
    CLDT_ERR_NO_SPACE = -2,
    /* Bytes or state violate the current protocol or data-structure contract. */
    CLDT_ERR_MALFORMED = -3,
    /* Frame version cannot be safely interpreted by this implementation. */
    CLDT_ERR_UNSUPPORTED_VERSION = -4,
    /* Required message authenticity verification failed. */
    CLDT_ERR_AUTHENTICATION = -5,
    /* A finite deadline or policy TTL elapsed before acceptable processing. */
    CLDT_ERR_EXPIRED = -6,
    /* A logical item or policy epoch has already been accepted. */
    CLDT_ERR_DUPLICATE = -7,
    /* A record or epoch is older than the accepted monotonic sequence. */
    CLDT_ERR_OUT_OF_ORDER = -8,
    /* A syntactically valid value exceeds a declared or compiled safety bound. */
    CLDT_ERR_OUT_OF_RANGE = -9,
    /* The operation conflicts with lifecycle or ownership state. */
    CLDT_ERR_WRONG_STATE = -10,
    /* Required attachment, calibration, or evidence is not yet available. */
    CLDT_ERR_NOT_READY = -11,
    /* An external file, socket, broker, or platform operation failed. */
    CLDT_ERR_IO = -12,
    /* A well-formed observation or command is older than its freshness rule. */
    CLDT_ERR_STALE = -13,
    /* A well-formed frame belongs to a run other than the active run. */
    CLDT_ERR_WRONG_RUN = -14,
    /* Command coordinator identity differs from the commissioned authority. */
    CLDT_ERR_WRONG_AUTHORITY = -15,
    /* Intentional scaffold marker; it is never a valid experiment outcome. */
    CLDT_ERR_NOT_IMPLEMENTED = -127
} cldt_status_t;

#ifdef __cplusplus
}
#endif

#endif
~~~

### **`common/include/cldt/cldt_types.h`**

Pekerjaan pada header ini: bekukan arti normatif `flags`, `hop_limit`, nilai not - applicable, serta encoding `detail` untuk event yang dipakai. Identity dan nama field yang sudah dideklarasikan menjadi vocabulary bersama bagi source, test, manifest, dan evidence.

~~~c
#ifndef CLDT_TYPES_H
#define CLDT_TYPES_H

#include <stdbool.h>
#include <stddef.h>
#include <stdint.h>

#ifdef __cplusplus
extern "C" {
#endif

#define CLDT_PROTOCOL_MAGIC UINT16_C(0x434C)
#define CLDT_PROTOCOL_VERSION UINT8_C(1)
#define CLDT_WIRE_HEADER_BYTES UINT16_C(72)
#define CLDT_MAX_PAYLOAD_BYTES UINT16_C(256)
#define CLDT_AUTH_TAG_BYTES 16U
#define CLDT_TRACE_DETAIL_BYTES 24U
#define CLDT_POLICY_STREAM_COUNT 4U
#define CLDT_COMMAND_AUTHORITY_NODE_ID UINT32_C(0)

typedef uint32_t cldt_node_id_t;
typedef uint64_t cldt_run_id_t;
typedef uint32_t cldt_boot_id_t;
typedef uint32_t cldt_sequence_t;
typedef uint32_t cldt_policy_epoch_t;

typedef enum {
    CLDT_NODE_GATEWAY_HOST = 0,
    CLDT_NODE_RADIO_COPROCESSOR,
    CLDT_NODE_ROUTER_ENDPOINT,
    CLDT_NODE_LOW_POWER_ENDPOINT
} cldt_node_role_t;

typedef enum {
    CLDT_TRAFFIC_CONTROL = 0,
    CLDT_TRAFFIC_CRITICAL,
    CLDT_TRAFFIC_TELEMETRY,
    CLDT_TRAFFIC_BULK,
    CLDT_TRAFFIC_COUNT
} cldt_traffic_class_t;

typedef enum {
    CLDT_FRAME_OBSERVATION = 0,
    CLDT_FRAME_COMMAND,
    CLDT_FRAME_ACKNOWLEDGEMENT,
    CLDT_FRAME_CLOCK_SYNC,
    CLDT_FRAME_HEALTH
} cldt_frame_kind_t;

typedef enum {
    CLDT_EVENT_TASK_RELEASE = 0,
    CLDT_EVENT_TASK_START,
    CLDT_EVENT_TASK_FINISH,
    CLDT_EVENT_TASK_BLOCK,
    CLDT_EVENT_QUEUE_ENQUEUE,
    CLDT_EVENT_QUEUE_DEQUEUE,
    CLDT_EVENT_QUEUE_REJECT,
    CLDT_EVENT_POOL_EXHAUSTION,
    CLDT_EVENT_MESSAGE_SEND,
    CLDT_EVENT_MESSAGE_ACK,
    CLDT_EVENT_MESSAGE_EXPIRE,
    CLDT_EVENT_MESSAGE_COALESCE,
    CLDT_EVENT_MESSAGE_DROP,
    CLDT_EVENT_MESSAGE_DUPLICATE,
    CLDT_EVENT_LINK_CHANGE,
    CLDT_EVENT_POWER_SAMPLE,
    CLDT_EVENT_POLICY_APPLY,
    CLDT_EVENT_POLICY_REJECT,
    CLDT_EVENT_POLICY_FALLBACK,
    CLDT_EVENT_HEALTH,
    CLDT_EVENT_COUNT
} cldt_event_kind_t;

typedef enum {
    CLDT_GATE_COLD = 0,
    CLDT_GATE_OBSERVE,
    CLDT_GATE_TRUSTED,
    CLDT_GATE_ABSTAIN
} cldt_gate_state_t;

typedef enum {
    CLDT_MODEL_NAIVE = 0,
    CLDT_MODEL_NETWORK_ONLY,
    CLDT_MODEL_CROSS_LAYER,
    CLDT_MODEL_VARIANT_COUNT
} cldt_model_variant_t;

/*
 * In-memory metadata. It is not a packed wire structure. Encoding and decoding
 * must be performed field by field through cldt_protocol.h.
 *
 * Identity is frame-kind specific. For observations, acknowledgements, health,
 * and trace-bearing frames, node_id/boot_id identify the emitting device. A
 * version 1 command is one global policy datagram for every endpoint admitted
 * to the run: node_id is CLDT_COMMAND_AUTHORITY_NODE_ID and boot_id identifies
 * the host coordinator process that issued it, not a destination. The gateway
 * guards and forwards those identical bytes. Version 1 does not define
 * different authenticated command bytes per endpoint.
 */
typedef struct {
    cldt_frame_kind_t kind;
    cldt_traffic_class_t traffic_class;
    uint16_t flags;
    uint8_t hop_limit;
    cldt_node_id_t node_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t sequence;
    cldt_policy_epoch_t policy_epoch;
    cldt_run_id_t run_id;
    uint64_t transmit_local_us;
    uint64_t deadline_local_us;
} cldt_frame_meta_t;

/*
 * Decoder output borrows payload memory from the input byte buffer. The caller
 * must keep that buffer alive and unchanged while this view is in use.
 */
typedef struct {
    cldt_frame_meta_t meta;
    const uint8_t *payload;
    uint16_t payload_bytes;
    uint32_t crc32c;
    uint8_t authentication_tag[CLDT_AUTH_TAG_BYTES];
} cldt_frame_view_t;

typedef struct {
    cldt_event_kind_t kind;
    /* Every work-item event carries its class; HEALTH may use CLDT_TRAFFIC_COUNT. */
    cldt_traffic_class_t traffic_class;
    cldt_node_id_t node_id;
    cldt_boot_id_t boot_id;
    cldt_sequence_t sequence;
    cldt_policy_epoch_t policy_epoch;
    cldt_run_id_t run_id;
    uint64_t local_time_us;
    /*
     * Work-item events repeat the item's release and absolute deadline in the
     * same local monotonic clock domain as local_time_us. Non-work-item events
     * store zero in both fields. This permits stateless aggregate timing while
     * preserving raw timestamps for a separate per-item lifecycle audit.
     */
    uint64_t release_local_us;
    uint64_t deadline_local_us;
    uint32_t task_id;
    int8_t core_id;
    uint16_t queue_depth;
    int16_t link_rssi_dbm;
    uint32_t time_uncertainty_us;
    /* Fixed-size auxiliary bytes; each event kind documents its own encoding. */
    uint8_t detail[CLDT_TRACE_DETAIL_BYTES];
} cldt_trace_record_t;

typedef struct {
    uint32_t release_period_ms[CLDT_POLICY_STREAM_COUNT];
    uint32_t phase_offset_ms[CLDT_POLICY_STREAM_COUNT];
    uint16_t burst_limit[CLDT_POLICY_STREAM_COUNT];
    uint16_t batch_size[CLDT_POLICY_STREAM_COUNT];
    uint32_t token_rate_milli_pps[CLDT_POLICY_STREAM_COUNT];
    cldt_policy_epoch_t epoch;
    uint64_t issued_gateway_us;
    uint32_t ttl_ms;
} cldt_policy_t;

typedef struct {
    cldt_model_variant_t model_variant;
    uint64_t model_revision;
    uint64_t horizon_start_host_us;
    uint64_t horizon_end_host_us;
    uint64_t evaluated_host_us;
    uint64_t newest_observation_host_us;
    uint32_t sample_count;
    uint32_t model_lag_us;
    uint32_t clock_uncertainty_us;
    double relative_p95_error;
    double pdr_error_points;
    double prediction_interval_coverage;
    /* False when required horizon evidence is missing, stale, or unreconciled. */
    bool observation_integrity_valid;
    bool inside_calibrated_region;
} cldt_fidelity_sample_t;

#ifdef __cplusplus
}
#endif

#endif
~~~

### **`common/include/cldt/cldt_protocol.h`**

Pekerjaan pada header ini: bekukan CRC coverage, authentication coverage, legal value/range, dan perilaku kegagalan untuk setiap frame kind. Offset wire yang sudah ada menjadi satu - satunya layout yang dipakai oleh fixed vectors.

~~~c
#ifndef CLDT_PROTOCOL_H
#define CLDT_PROTOCOL_H

#include <stddef.h>
#include <stdint.h>

#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"

#ifdef __cplusplus
extern "C" {
#endif

typedef cldt_status_t (*cldt_authenticate_fn)(
    void *context,
    const uint8_t *bytes,
    size_t byte_count,
    uint8_t output_tag[CLDT_AUTH_TAG_BYTES]);

typedef struct {
    cldt_authenticate_fn calculate_tag;
    void *context;
} cldt_authenticator_t;

/*
 * Version 1 frame layout. Every multi-byte integer uses network byte order.
 * The two reserved bytes are transmitted as zero and rejected when nonzero.
 * These offsets are the wire contract; sizeof(cldt_frame_meta_t) is not.
 */
#define CLDT_WIRE_MAGIC_OFFSET 0U
#define CLDT_WIRE_VERSION_OFFSET 2U
#define CLDT_WIRE_KIND_OFFSET 3U
#define CLDT_WIRE_TRAFFIC_CLASS_OFFSET 4U
#define CLDT_WIRE_FLAGS_OFFSET 5U
#define CLDT_WIRE_HOP_LIMIT_OFFSET 7U
#define CLDT_WIRE_NODE_ID_OFFSET 8U
#define CLDT_WIRE_BOOT_ID_OFFSET 12U
#define CLDT_WIRE_SEQUENCE_OFFSET 16U
#define CLDT_WIRE_POLICY_EPOCH_OFFSET 20U
#define CLDT_WIRE_RUN_ID_OFFSET 24U
#define CLDT_WIRE_TRANSMIT_LOCAL_US_OFFSET 32U
#define CLDT_WIRE_DEADLINE_LOCAL_US_OFFSET 40U
#define CLDT_WIRE_PAYLOAD_BYTES_OFFSET 48U
#define CLDT_WIRE_RESERVED_OFFSET 50U
#define CLDT_WIRE_RESERVED_BYTES 2U
#define CLDT_WIRE_CRC32C_OFFSET 52U
#define CLDT_WIRE_AUTH_TAG_OFFSET 56U

/*
 * Version 1 policy payload layout. Array elements are contiguous and encoded
 * in traffic-class order from CLDT_TRAFFIC_CONTROL through CLDT_TRAFFIC_BULK.
 * The policy epoch in this payload must equal the frame metadata epoch.
 */
#define CLDT_POLICY_WIRE_RELEASE_PERIOD_OFFSET 0U
#define CLDT_POLICY_WIRE_PHASE_OFFSET 16U
#define CLDT_POLICY_WIRE_BURST_LIMIT_OFFSET 32U
#define CLDT_POLICY_WIRE_BATCH_SIZE_OFFSET 40U
#define CLDT_POLICY_WIRE_TOKEN_RATE_OFFSET 48U
#define CLDT_POLICY_WIRE_EPOCH_OFFSET 64U
#define CLDT_POLICY_WIRE_ISSUED_GATEWAY_US_OFFSET 68U
#define CLDT_POLICY_WIRE_TTL_MS_OFFSET 76U
#define CLDT_POLICY_WIRE_BYTES 80U

#if (CLDT_WIRE_AUTH_TAG_OFFSET + CLDT_AUTH_TAG_BYTES) != CLDT_WIRE_HEADER_BYTES
#error "Version 1 frame offsets do not match CLDT_WIRE_HEADER_BYTES"
#endif

#if CLDT_POLICY_STREAM_COUNT != 4U
#error "Version 1 policy layout requires exactly four traffic classes"
#endif

#if (CLDT_POLICY_WIRE_TTL_MS_OFFSET + 4U) != CLDT_POLICY_WIRE_BYTES
#error "Version 1 policy offsets do not match CLDT_POLICY_WIRE_BYTES"
#endif

/*
 * Returns the exact output size required for this payload. Zero means the
 * payload cannot be represented by the current protocol version.
 */
size_t cldt_protocol_encoded_size(size_t payload_bytes);

/*
 * Encodes one frame into caller-owned storage. No heap allocation or I/O is
 * permitted. output_bytes is written only on success.
 */
cldt_status_t cldt_protocol_encode(
    const cldt_frame_meta_t *meta,
    const uint8_t *payload,
    size_t payload_bytes,
    const cldt_authenticator_t *authenticator,
    uint8_t *output,
    size_t output_capacity,
    size_t *output_bytes);

/*
 * Decodes and validates one complete datagram. The returned payload view
 * borrows memory from input. The function must reject trailing bytes,
 * truncation, an unsupported version, CRC failure, and required-auth failure.
 */
cldt_status_t cldt_protocol_decode(
    const uint8_t *input,
    size_t input_bytes,
    const cldt_authenticator_t *authenticator,
    bool authentication_required,
    cldt_frame_view_t *output_view);

/*
 * Applies freshness and ordering checks after successful decoding. Times are
 * in the gateway monotonic domain; uncertainty expands the rejection margin.
 * On an endpoint, applied_epoch must be the RAM mirror of a valid durable
 * replay record. The caller still owns issuer validation, durable advancement,
 * local limits, and atomic policy publication.
 */
cldt_status_t cldt_protocol_validate_command(
    const cldt_frame_view_t *frame,
    cldt_run_id_t active_run_id,
    cldt_policy_epoch_t applied_epoch,
    uint64_t now_gateway_us,
    uint32_t time_uncertainty_us);

#ifdef __cplusplus
}
#endif

#endif
~~~

### **`common/include/cldt/cldt_auth.h`**

Pekerjaan pada header ini: tentukan ownership dan lifetime `cldt_auth_context_t`, sumber key yang diizinkan, zeroization, serta perilaku setelah kegagalan. Keputusan ini harus konsisten dengan `SECURITY.md`; implementasi command authentication sendiri belum menjadi deliverable Day 10.

~~~c
#ifndef CLDT_AUTH_H
#define CLDT_AUTH_H

#include <stddef.h>
#include <stdint.h>
#include <stdbool.h>
#include "cldt/cldt_status.h"
#include "cldt/cldt_types.h"
#include "cldt/cldt_protocol.h"

#define CLDT_AUTH_KEY_BYTES 32
#define CLDT_AUTH_NONCE_BYTES 12

typedef struct {
    uint8_t key[CLDT_AUTH_KEY_BYTES];
    bool key_loaded;
} cldt_auth_context_t;

cldt_status_t cldt_auth_init(cldt_auth_context_t *ctx);
cldt_status_t cldt_auth_load_key(cldt_auth_context_t *ctx, const uint8_t key[CLDT_AUTH_KEY_BYTES]);
void cldt_auth_build_nonce(cldt_run_id_t run_id, cldt_policy_epoch_t epoch, uint8_t nonce[CLDT_AUTH_NONCE_BYTES]);
cldt_status_t cldt_auth_sign(cldt_auth_context_t *ctx, cldt_run_id_t run_id, cldt_policy_epoch_t epoch, const uint8_t *aad, size_t aad_len, const uint8_t *payload, size_t payload_len, uint8_t tag[CLDT_AUTH_TAG_BYTES]);
cldt_status_t cldt_auth_verify(cldt_auth_context_t *ctx, cldt_run_id_t run_id, cldt_policy_epoch_t epoch, const uint8_t *aad, size_t aad_len, const uint8_t *payload, size_t payload_len, const uint8_t tag[CLDT_AUTH_TAG_BYTES]);
cldt_authenticator_t cldt_auth_as_authenticator(cldt_auth_context_t *ctx);

#endif // CLDT_AUTH_H
~~~

### **Contract decisions yang ditutup pada Days 1-3**

1. Pada `cldt_crc32c.h`, convention seed, final XOR, empty input, dan incremental call dibekukan sebelum source/test diisi.
2. Pada `cldt_types.h`/`cldt_protocol.h`, arti setiap bit `flags`, legal `hop_limit`, zero/not - applicable fields, CRC coverage, auth coverage, dan frame - kind authentication matrix ditulis normatif.
3. Pada `cldt_auth.h`, lifetime `cldt_auth_context_t`, zeroization, caller ownership, allowed key source, dan behavior setelah failure diputuskan. Endpoint/gateway provisioning path harus konsisten dengan `SECURITY.md`.
4. Jika keputusan memerlukan perubahan struct/signature, header diubah bersama source, tests, `DESIGN.md`, dan `SECURITY.md`. Existing declarations tidak dipertahankan hanya karena sudah bisa dikompilasi.


## **Step 01: Membekukan Fondasi yang Benar - Benar Dipakai**
**Track:** **`Scope - Hardware - Toolchain`**

Commit project yang menjadi dasar panduan ini adalah `0dee73247d4e87a3afec8fb90f37439cbb96523b`. Yang belum diketahui berasal dari dunia fisik: revisi board yang datang, pemetaan port serial, kondisi kabel, commit ESP - IDF yang dipilih, serta hash image hasil build. Kolom contoh di bawah hanya menunjukkan bentuk isian. Setiap contoh diganti dengan hasil yang terlihat dari board, command, atau file build milik sesi kerja sendiri.

### **Lingkungan kerja**

| Lingkungan | Pemakaian pada fase ini | Catatan |
| - | - | - |
| Terminal lokal Windows/WSL | Git, CMake/CTest, hash file, penyimpanan log | CMake minimal 3.20 dan compiler C11 sesuai `CONTRIBUTING.md` |
| Shell ESP - IDF lokal | build, flash, `menuconfig`, dan serial monitor | revision ESP - IDF dibekukan satu kali dan dicatat |
| Meja hardware | label board, daya, UART, power - cycle, Thread, UDP | tiga board dipakai untuk gate; C6 kedua belum dipasang |
| Google Colab | tidak diperlukan | Colab tidak menjadi jalur flash/USB/UART; analisis Python baru relevan setelah recorder Week 3 |

### **Board minimum untuk gate**

| Label fisik | Board | Fungsi pada fase ini |
| - | - | - |
| gateway | ESP32 - S3 N16R8 | menjalankan contoh upstream `ot_br` |
| rcp | XIAO ESP32 - C6 | menjalankan contoh upstream `ot_rcp` |
| endpoint - a | XIAO ESP32 - C6 | menjalankan contoh upstream `ot_cli` |
| endpoint - b | XIAO ESP32 - C6 | tetap disimpan; baru dipasang setelah gate lulus |

BOM menetapkan ceiling Rp 1.750.000, committed plan Rp 1.575.000, dan reserve Rp 175.000. Sensor, INA219, sniffer, serta node tambahan bukan pembelian fase ini. Logic analyzer adalah item pertama yang boleh dilepas jika board atau powered hub wajib tidak muat anggaran.

### **Urutan kerja**

1. Board dan kabel diberi label permanen sebelum lebih dari satu perangkat disambungkan. Setiap kabel diuji sebagai kabel data, bukan hanya kabel charge.
2. Satu board dipasang bergantian untuk memetakan label fisik ke port serial. Chip, flash, dan board revision disimpan dari log nyata.
3. Checkout ESP - IDF dipilih, lalu commit penuh dan versi tool dicatat. Source ESP - IDF tidak disalin ke repository ini.
4. Kontrak wiring dicocokkan dengan board yang benar - benar datang: S3 `GPIO17/TX` menuju C6 `D7/GPIO17/RX`; S3 `GPIO18/RX` menerima C6 `D6/GPIO16/TX`; ground disatukan lebih dulu.
5. Reset dan boot control tetap tidak tersambung. Keduanya baru dipakai bila perilaku elektrisnya sudah diperiksa.
6. Byte offsets pada `cldt_protocol.h`, pilihan CRC - 32C, dan ChaCha20 - Poly1305 pada `cldt_auth.h` dibekukan sebagai kontrak. Freeze berarti tidak mengubah desain saat hasil datang; bukan berarti encode/auth harus selesai sekarang.

~~~powershell
git rev-parse --show-toplevel
git rev-parse HEAD
git branch --show-current
git status --short
git remote -v
git submodule status

python --version
cmake --version
idf.py --version
git -C $env:IDF_PATH rev-parse HEAD
~~~

### **Catatan fisik dan toolchain**

| Catatan | Contoh format - ganti dengan hasil aktual | Diambil dari |
| - | - | - |
| Revisi board gateway | e.g. ESP32 - S3 N16R8 dan revision yang tercetak pada PCB | silkscreen, invoice, dan chip log |
| Revisi board RCP | e.g. XIAO ESP32 - C6 dan revision yang tercetak pada PCB | silkscreen, invoice, dan chip log |
| Revisi board endpoint A | e.g. XIAO ESP32 - C6 dan revision yang tercetak pada PCB | silkscreen, invoice, dan chip log |
| Port serial gateway | e.g. COM7 | enumerasi saat hanya gateway terhubung |
| Port serial RCP | e.g. COM8 | enumerasi saat hanya RCP terhubung |
| Port serial endpoint A | e.g. COM9 | enumerasi saat hanya endpoint terhubung |
| Commit ESP - IDF | e.g. 40 karakter hexadecimal dari output command | `git - C $env:IDF_PATH rev - parse HEAD` |
| Versi ESP - IDF | e.g. ESP - IDF v5.2.x | `idf.py - version` |
| Versi CMake | e.g. cmake version 3.30.0 | `cmake - version` setelah tersedia |
| Versi compiler | e.g. gcc 14.2.0 atau MSVC 19.xx | compiler yang benar - benar dipakai |


## **Step 02: Membedakan Build Scaffold, Skip, dan Implementasi**
**Track:** **`Host - CI Contract`**

Root `CMakeLists.txt` membangun `cldt_common` dan `cldt_host`; test hanya ikut saat `CLDT_BUILD_TESTS=ON`. `tests/CMakeLists.txt` sudah mendaftarkan tujuh target dan menetapkan `SKIP_RETURN_CODE 77`. Karena itu tidak ada alasan membuat file test atau target CMake baru untuk fase ini.

### **File build yang terhubung**

Root build, daftar source common, dan daftar target test berikut adalah contract yang dipakai oleh command pada cell ini. Tidak ada target baru yang perlu dibuat untuk CRC.

#### `CMakeLists.txt`

~~~cmake
cmake_minimum_required(VERSION 3.20)

project(cldt_host LANGUAGES C)

option(CLDT_BUILD_TESTS "Build the host-side skeletal tests" OFF)

set(CMAKE_C_STANDARD 11)
set(CMAKE_C_STANDARD_REQUIRED ON)
set(CMAKE_C_EXTENSIONS OFF)

add_subdirectory(common)
add_subdirectory(host)

if(CLDT_BUILD_TESTS)
    enable_testing()
    add_subdirectory(tests)
endif()

~~~

#### `common/CMakeLists.txt`

~~~cmake
set(CLDT_COMMON_SOURCES
    "src/cldt_protocol.c"
    "src/cldt_clock_sync.c"
    "src/cldt_control_profile.c"
    "src/cldt_metrics.c"
    "src/cldt_event_trace.c"
    "src/cldt_auth.c"
    "src/cldt_crc32c.c"
)

if(COMMAND idf_component_register)
    idf_component_register(
        SRCS ${CLDT_COMMON_SOURCES}
        INCLUDE_DIRS "include"
    )
else()
    add_library(cldt_common STATIC ${CLDT_COMMON_SOURCES})
    target_include_directories(cldt_common
        PUBLIC
            ${CMAKE_CURRENT_SOURCE_DIR}/include
    )

    if(MSVC)
        target_compile_options(cldt_common PRIVATE /W4)
    else()
        target_compile_options(cldt_common PRIVATE -Wall -Wextra -Wpedantic -Wconversion)
    endif()
endif()

~~~

#### `tests/CMakeLists.txt`

~~~cmake
function(cldt_add_skeletal_test name source)
    add_executable(${name} ${source})
    target_link_libraries(${name} PRIVATE cldt_common)
    add_test(NAME ${name} COMMAND ${name})
    set_tests_properties(${name} PROPERTIES SKIP_RETURN_CODE 77)
endfunction()

cldt_add_skeletal_test(test_protocol test_protocol.c)
cldt_add_skeletal_test(test_crc32c test_crc32c.c)
cldt_add_skeletal_test(test_auth test_auth.c)
cldt_add_skeletal_test(test_clock_sync test_clock_sync.c)
cldt_add_skeletal_test(test_metrics test_metrics.c)
cldt_add_skeletal_test(test_event_trace test_event_trace.c)
cldt_add_skeletal_test(test_control_profile test_control_profile.c)

~~~

~~~powershell
cmake -S . -B build -DCLDT_BUILD_TESTS=ON
cmake --build build --parallel
ctest --test-dir build --output-on-failure
~~~

Makna hasil awalnya:

1. Configure dan build yang bersih membuktikan header, source list, serta signature saat ini konsisten.
2. Tujuh baris “Skipped” adalah kondisi scaffold yang memang didokumentasikan di `CONTRIBUTING.md`.
3. Satu test baru boleh berubah dari skip menjadi pass setelah seluruh checklist pada komentar test tersebut benar - benar dilaksanakan.
4. Stub lain yang masih mengembalikan `CLDT_ERR_NOT_IMPLEMENTED` tidak boleh diklaim selesai hanya karena linker berhasil.
5. `cldt_host` sendiri sengaja keluar gagal karena manifest loading, recording, model, dan coordination belum diimplementasikan.

Schema dan delapan strict manifest diperiksa terpisah. Hasil command dicatat pada tabel di bawah; manifest tetap `state: "template"` sampai keputusan eksperimennya benar - benar tersedia.

~~~powershell
python -m pip install "jsonschema==4.26.0"
python -c "import json,pathlib; from jsonschema import Draft202012Validator; s=json.loads(pathlib.Path('schemas/experiment.schema.json').read_text()); Draft202012Validator.check_schema(s); v=Draft202012Validator(s); files=sorted(pathlib.Path('experiments').glob('*.json')); assert all(not list(v.iter_errors(json.loads(p.read_text()))) for p in files); print(f'validated {len(files)} strict manifests')"
~~~

### **Workflow yang benar - benar dijalankan: `.github/workflows/scaffold - validation.yml`**

Workflow ini hanya configure/build host scaffold, menjalankan CTest, dan memvalidasi 8 strict JSON + 8 JSONC companions. Ia tidak membangun atau mem - flash endpoint/gateway/RCP, tidak menjalankan hardware, dan tujuh test masih dapat berakhir SKIPPED melalui code 77.

~~~yaml
name: Scaffold Validation

on:
  push:
    branches:
      - main
  pull_request:

permissions:
  contents: read

jobs:
  host-and-manifests:
    name: Host Build and Manifest Contracts
    runs-on: ubuntu-latest

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Configure host scaffold
        run: cmake -S . -B build -DCLDT_BUILD_TESTS=ON

      - name: Build host scaffold
        run: cmake --build build --parallel

      - name: Run test harness
        run: ctest --test-dir build --output-on-failure

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      - name: Install JSON Schema validator
        run: python -m pip install --disable-pip-version-check "jsonschema==4.26.0"

      - name: Validate experiment manifests
        shell: bash
        run: |
          python - <<'PY'
          import json
          from pathlib import Path

          from jsonschema import Draft202012Validator

          schema_path = Path("schemas/experiment.schema.json")
          schema = json.loads(schema_path.read_text(encoding="utf-8"))
          Draft202012Validator.check_schema(schema)
          validator = Draft202012Validator(schema)

          strict_files = sorted(Path("experiments").glob("*.json"))
          if not strict_files:
              raise SystemExit("No strict experiment manifests were found.")

          strict_ids = {}
          for path in strict_files:
              document = json.loads(path.read_text(encoding="utf-8"))
              errors = sorted(validator.iter_errors(document), key=lambda error: list(error.path))
              if errors:
                  for error in errors:
                      location = "/" + "/".join(str(part) for part in error.path)
                      print(f"{path}:{location}: {error.message}")
                  raise SystemExit(f"Schema validation failed for {path}.")
              strict_ids[path.stem] = document["experiment_id"]

          authoring_files = sorted(Path("experiments/authoring").glob("*.jsonc"))
          if len(authoring_files) != len(strict_files):
              raise SystemExit("Strict JSON and JSONC authoring manifest counts differ.")

          for path in authoring_files:
              uncommented = "\n".join(
                  line for line in path.read_text(encoding="utf-8").splitlines()
                  if not line.lstrip().startswith("//")
              )
              document = json.loads(uncommented)
              if document["experiment_id"] != strict_ids.get(path.stem):
                  raise SystemExit(f"Experiment ID mismatch for {path}.")

          print(f"Validated {len(strict_files)} strict manifests and {len(authoring_files)} authoring copies.")
          PY
~~~

### **Rekaman build lokal**

| Catatan | Contoh format - ganti dengan hasil aktual |
| - | - |
| Exit code configure CMake | e.g. 0 |
| Exit code build | e.g. 0 |
| Test berstatus passed | e.g. 0 sebelum CRC selesai; 1 setelah `test_crc32c` benar - benar pass |
| Test berstatus skipped | e.g. 7 pada scaffold awal; 6 setelah satu skip ditutup |
| Test berstatus failed | e.g. 0 |
| Strict manifest tervalidasi | e.g. 8 |
| Lokasi log build | e.g. C:\logs\cldt\host - build.txt |

Jika `cmake` tidak ditemukan, pekerjaan pertama hanya menyediakan CMake ≥3.20 dan compiler C11 pada terminal yang dipakai. Mengubah source untuk menyiasati tool yang belum terpasang akan menghasilkan diagnosis yang salah.


## **Step 03: Menutup TODO CRC - 32C yang Memang Ada di Repo**
**Track:** **`Common - Fixed Vector`**

CRC - 32C dipilih sebagai pekerjaan kode minimum karena ia portable, berdiri sendiri, secara eksplisit meminta fixed vectors, dan dapat mengganti satu skip tanpa berpura - pura bahwa protocol CLDT sudah selesai. Scope edit - nya hanya `common/src/cldt_crc32c.c` dan `tests/test_crc32c.c`. Header yang terhubung tidak diabaikan: signature, include guard, dan jenis integer berikut harus tetap cocok dengan source dan test.

### **`common/include/cldt/cldt_crc32c.h`**

~~~c
#ifndef CLDT_CRC32C_H
#define CLDT_CRC32C_H

#include <stddef.h>
#include <stdint.h>

#ifdef __cplusplus
extern "C" {
#endif

/*
 * Planned platform-neutral CRC-32C (Castagnoli) contract. The scaffold returns
 * a placeholder value until fixed known-answer vectors are implemented. Do not
 * substitute esp_rom_crc32_le without proving the polynomial and seed/final-XOR
 * conventions match CRC-32C on every target.
 */
uint32_t cldt_crc32c(uint32_t seed, const void *buf, size_t len);

#ifdef __cplusplus
}
#endif

#endif

~~~

Berikut scaffold source persis seperti repository. Badan fungsi tetap kosong secara semantik dan masih mengembalikan placeholder.

~~~c
#include "cldt/cldt_crc32c.h"

uint32_t cldt_crc32c(uint32_t seed, const void *buf, size_t len)
{
    /*
     * IMPLEMENTATION TODO:
     * - implement reflected CRC-32C using polynomial 0x82F63B78;
     * - freeze the seed and final-XOR convention with fixed vectors;
     * - verify the same vectors on the host, ESP32-S3, and ESP32-C6;
     * - compute the wire CRC over the canonical header-plus-payload sequence;
     * - do not use the similarly named ROM IEEE CRC-32 routine as CRC-32C.
     */
    (void)seed;
    (void)buf;
    (void)len;
    return UINT32_C(0);
}
~~~

Pekerjaan di fungsi tersebut terdiri dari keputusan dan implementasi berikut:

1. Konvensi `seed` dan final XOR ditulis lebih dahulu sebagai kontrak. API menerima `seed`, sehingga test incremental harus memakai state lanjutan dengan cara yang konsisten.
2. Algoritma yang digunakan adalah reflected CRC - 32C/Castagnoli dengan polynomial `0x82F63B78`, bukan CRC - 32/IEEE yang namanya mirip.
3. Input diproses sebagai byte, kemudian delapan langkah reduksi bit dilakukan untuk tiap byte. Tidak ada heap allocation.
4. Perilaku `len == 0` dibekukan melalui known - answer vector. Pemanggilan dengan pointer tidak valid dan panjang nonzero tidak diuji dengan dereference spekulatif.
5. Sequence untuk frame nanti adalah header canonical bytes 0-51 yang langsung diikuti payload; slot CRC dan auth tag tidak ikut dihitung.

Test scaffold yang benar juga sudah ada. Tidak ada file test ukuran terpisah dan tidak perlu menambah target CMake.

~~~c
#include <stdio.h>

#include "cldt/cldt_crc32c.h"

int main(void)
{
    /*
     * IMPLEMENTATION TODO:
     * 1. Freeze standard CRC-32C known-answer vectors, including empty input and
     *    "123456789", with the exact seed and final-XOR convention.
     * 2. Test incremental versus single-buffer updates, zero length, unaligned
     *    input, binary zero bytes, and the canonical header-plus-payload wire
     *    integrity sequence.
     * 3. Run identical vectors on the host, ESP32-S3, and ESP32-C6. A platform
     *    helper with the IEEE polynomial must fail the Castagnoli vector.
     * 4. Do not enable frame acceptance until this test is no longer skipped.
     */
    fprintf(stderr, "SKIP: CRC-32C tests have not been implemented.\n");
    return 77;
}
~~~

Urutan pengisiannya:

1. Known - answer values diambil dari rujukan CRC - 32C independen dan derivasi singkat disimpan di komentar test.
2. Assertion pertama mencakup empty input dan string `"123456789"`.
3. Data yang sama dihitung sekali sebagai satu buffer dan sekali dalam beberapa potongan; hasilnya harus identik.
4. Buffer dengan byte `0x00`, offset tidak sejajar, serta panjang nol diuji terpisah.
5. Satu negative control memakai helper/polynomial IEEE dan harus berbeda dari vector Castagnoli.
6. `return 77` baru diganti dengan exit sukses setelah seluruh butir komentar di atas ada dan benar. Mengganti kode keluar lebih awal hanya mengubah label CTest, bukan kualitas implementasi.
7. Vector yang sama nanti dijalankan pada build ESP32 - S3 dan ESP32 - C6; hasil platform dicatat, bukan diasumsikan dari host.

### **Hasil yang dicatat setelah dikerjakan**

| Pemeriksaan | Contoh format - ganti dengan hasil aktual |
| - | - |
| Rujukan konvensi CRC - 32C | e.g. judul rujukan, URL/nomor bagian, nilai seed, dan final XOR |
| Empty - input vector | e.g. PASS - expected dan actual ditulis dalam hexadecimal |
| `"123456789"` known - answer vector | e.g. PASS - expected dan actual ditulis dalam hexadecimal |
| Incremental versus single - buffer | e.g. PASS - kedua hasil sama |
| Negative control CRC - 32/IEEE | e.g. PASS - hasil berbeda dari Castagnoli |
| Exit code `test_crc32c` pada host | e.g. 0 |
| Vector yang sama pada ESP32 - S3 | e.g. PASS - catat expected, actual, dan log |
| Vector yang sama pada ESP32 - C6 | e.g. PASS - catat expected, actual, dan log |


## **Step 04: Membangun Radio Co - Processor Tanpa Project Logic**
**Track:** **`ESP32-C6 - Upstream RCP`**

RCP Day 10 bukan `firmware/endpoint/` dan bukan `firmware/gateway/`. Repository meminta contoh `openthread/ot_rcp` dari checkout ESP - IDF yang dibekukan, tanpa menyisipkan workload CLDT. Pemisahan ini membuat kegagalan awal dapat dilokalisasi ke board, image, UART, atau OpenThread sebelum source project ikut menambah variabel.

### **Langkah kerja**

1. C6 berlabel `rcp` disambungkan sendiri dan port - nya diidentifikasi dari perubahan daftar port sebelum/sesudah board dipasang.
2. Shell ESP - IDF dibuka dari checkout yang commit - nya sudah dicatat pada bagian 01.
3. Direktori contoh `$env:IDF_PATH\examples\openthread\ot_rcp` dipakai sebagai source build.
4. Target `esp32c6` dipilih. Transport dan pin UART dibaca dari konfigurasi contoh pada revision tersebut; nama Kconfig tidak ditebak dari revision lain.
5. Image dibangun, di - flash melalui USB board, lalu log flash disimpan.
6. `sdkconfig`, commit upstream, target, ukuran image, dan SHA - 256 binary disalin ke catatan bring - up lokal. Source upstream tidak dimasukkan ke repo CLDT.
7. Jika image gagal start, urutan diagnosisnya: target salah, flash/partition, USB cable, power, boot mode, baru setelah itu konfigurasi transport.

~~~powershell
Set-Location "$env:IDF_PATH\examples\openthread\ot_rcp"
idf.py set-target esp32c6
idf.py menuconfig
idf.py build
idf.py -p COMx flash
~~~

`COMx` adalah port yang berubah saat board `rcp` dipasang sendiri. Ia bukan angka contoh untuk disalin. Nama output binary diperoleh dari ringkasan `idf.py build`; file itulah yang di - hash.

~~~powershell
Get-FileHash -Algorithm SHA256 -LiteralPath "<path binary yang dilaporkan build>"
git -C $env:IDF_PATH rev-parse HEAD
~~~

### **Rekaman RCP**

| Catatan RCP | Contoh format - ganti dengan hasil aktual |
| - | - |
| Port serial USB | e.g. COM8 |
| Commit ESP - IDF | e.g. 40 karakter hexadecimal dari checkout yang dibangun |
| Target build | e.g. esp32c6 |
| Transport RCP | e.g. UART |
| Instance UART | e.g. UART1; salin dari konfigurasi build |
| GPIO TX RCP | e.g. 16; cocokkan dengan D6/GPIO16/TX |
| GPIO RX RCP | e.g. 17; cocokkan dengan D7/GPIO17/RX |
| Baud UART | e.g. 460800; salin nilai aktual dari konfigurasi kedua sisi |
| Flow control | e.g. disabled atau RTS/CTS; kedua sisi harus sama |
| SHA - 256 `sdkconfig` | e.g. 64 karakter hexadecimal dari `Get - FileHash` |
| SHA - 256 binary RCP | e.g. 64 karakter hexadecimal dari binary yang dilaporkan build |
| Lokasi log flash | e.g. C:\logs\cldt\rcp - flash.txt |


## **Step 05: Menghubungkan S3 ke RCP dengan Jalur yang Dapat Diprobe**
**Track:** **`ESP32-S3 - Border Router - Spinel`**

S3 menjalankan contoh upstream `openthread/ot_br`. Scaffold `firmware/gateway/` belum digunakan untuk gate ini: `app_main.c` sendiri menyatakan runtime, provisioning, Thread bridge, backhaul, dan policy guard belum diimplementasikan. Nilai default `GPIO18` untuk RX dan `GPIO17` untuk TX pada `firmware/gateway/main/Kconfig.projbuild` adalah kontrak project; konfigurasi upstream tetap harus dicocokkan pada revision ESP - IDF yang dipakai.

### **Wiring**

| Jalur | S3 | XIAO C6 RCP | Catatan |
| - | - | - | - |
| data ke RCP | `GPIO17/TX` | `D7/GPIO17/RX` | TX disilang ke RX |
| data dari RCP | `GPIO18/RX` | `D6/GPIO16/TX` | RX menerima TX |
| referensi | `GND` | `GND` | ground disatukan sebelum signal |
| reset | belum dipakai | `EN` | tetap terbuka |
| boot | belum dipakai | boot/`GPIO9` | tetap terbuka |

Kedua board mendapat power dari jalur yang terdokumentasi. Signal UART tidak dipakai sebagai sumber power. Kabel dibuat pendek; logic analyzer, bila dipakai, hanya memantau TX/RX dan ground tanpa mengubah level.

### **Langkah kerja**

1. RCP yang sudah di - flash dimatikan, ground dipasang, lalu jalur TX/RX disilang sesuai tabel.
2. S3 dipasang sendiri lebih dahulu untuk mengidentifikasi port USB - nya.
3. Contoh `$env:IDF_PATH\examples\openthread\ot_br` dibangun untuk `esp32s3`.
4. Pada `menuconfig`, external RCP/UART dipilih menggunakan opsi yang benar - benar tersedia pada revision yang dibekukan. Pin, baud, flow control, dan reset behavior harus sama dengan image RCP.
5. Credential Wi - Fi/backhaul, bila diperlukan oleh contoh, disimpan lokal dan tidak dicatat di notebook atau commit.
6. Setelah flash, log S3 diamati sampai host berhasil berbicara dengan RCP. Timeout Spinel dicatat sebagai kegagalan, bukan disembunyikan dengan reboot berulang.
7. Bila Spinel gagal, diagnosis bergerak dari power/ground → crossover TX/RX → baud/flow control → reset timing → capture UART. Project runtime belum disentuh pada tahap ini.

~~~powershell
Set-Location "$env:IDF_PATH\examples\openthread\ot_br"
idf.py set-target esp32s3
idf.py menuconfig
idf.py build
idf.py -p COMy flash monitor
~~~

`COMy` adalah port milik S3. Port USB C6 RCP tetap berbeda dari UART Spinel antara S3 dan C6.

### **Rekaman border router dan Spinel**

| Catatan border router/Spinel | Contoh format - ganti dengan hasil aktual |
| - | - |
| Port serial USB gateway | e.g. COM7 |
| Commit ESP - IDF | e.g. commit yang sama dengan build RCP |
| SHA - 256 `sdkconfig` gateway | e.g. 64 karakter hexadecimal dari `Get - FileHash` |
| SHA - 256 binary gateway | e.g. 64 karakter hexadecimal dari binary yang dilaporkan build |
| Baud Spinel | e.g. 460800; harus sama dengan konfigurasi RCP |
| Flow control Spinel | e.g. disabled atau RTS/CTS; harus sama dengan konfigurasi RCP |
| Waktu pertama komunikasi Spinel berhasil | e.g. 8.4 s setelah power - on |
| Jumlah timeout Spinel | e.g. 0 |
| Jumlah reset tidak direncanakan | e.g. 0 |
| Lokasi capture UART | e.g. C:\logs\cldt\spinel - session.sr |


## **Step 06: Membentuk Thread Minimum Sebelum Protocol CLDT**
**Track:** **`ESP32-C6 - One Endpoint - UDP`**

Endpoint pertama memakai contoh upstream `openthread/ot_cli`, bukan `firmware/endpoint/`. Ini konsisten dengan `README.md`: satu endpoint harus lolos UDP sebelum endpoint kedua atau project protocol masuk. Kalimat di `tests/test_protocol.c` yang melarang menghubungkan Thread/MQTT sebelum protocol test green berlaku pada integrasi frame CLDT, bukan smoke path upstream yang memang ditempatkan lebih awal oleh jadwal repository.

### **Build endpoint**

~~~powershell
Set-Location "$env:IDF_PATH\examples\openthread\ot_cli"
idf.py set-target esp32c6
idf.py menuconfig
idf.py build
idf.py -p COMz flash monitor
~~~

`COMz` berasal dari board `endpoint - a` yang dipasang sendiri. Image, `sdkconfig`, commit ESP - IDF, dan log flash disimpan seperti RCP.

### **Membentuk network**

Sintaks CLI diverifikasi melalui `help` pada image yang benar - benar di - flash karena command dapat berubah antar revision. Alur OpenThread yang diharapkan tetap sama:

~~~text
# Console border router / OpenThread leader
dataset init new
dataset commit active
ifconfig up
thread start
state
dataset active -x

# Console endpoint-a
dataset set active <active dataset dari console border router>
ifconfig up
thread start
state
ipaddr
~~~

Active dataset memuat network key. Hex mentahnya tidak dimasukkan ke Git, screenshot publik, atau notebook. Yang boleh dicatat adalah hash/identity yang tidak membuka credential, channel, PAN ID bila aman, dan lokasi evidence privat.

`state` pada endpoint dicatat sebagai hasil aktual. Label “router - capable” bukan bukti bahwa perangkat benar - benar menjadi router. Setelah attachment stabil, reachability diuji dua arah dengan alamat IPv6 yang dilaporkan oleh `ipaddr`.

### **UDP minimum**

Satu sisi membuka socket dan bind; sisi lain mengirim payload berurutan. Bentuk command umum berikut dicek lagi dengan `help udp` pada revision yang dipakai.

~~~text
# Endpoint-a
udp open
udp bind :: 1212

# Border router
udp open
udp send <alamat IPv6 endpoint-a> 1212 cldt-bringup-000001
~~~

Nomor sequence pada payload dinaikkan untuk tiap datagram. Dengan begitu duplicate, gap, dan reorder dapat terlihat tanpa memakai protocol CLDT.

### **Catatan attachment**

| Catatan attachment | Contoh format - ganti dengan hasil aktual |
| - | - |
| Port serial USB endpoint A | e.g. COM9 |
| SHA - 256 `sdkconfig` endpoint A | e.g. 64 karakter hexadecimal dari `Get - FileHash` |
| SHA - 256 binary endpoint A | e.g. 64 karakter hexadecimal dari binary yang dilaporkan build |
| `setup.thread_channel` | e.g. 15; salin channel dari active dataset |
| Hasil `state` pada border router | e.g. leader |
| Hasil `state` pada endpoint A | e.g. child atau router; catat yang benar - benar muncul |
| Alamat dari `ipaddr` yang dipakai untuk uji | e.g. fdxx:xxxx:xxxx:xxxx::abcd |
| Ping border router ke endpoint A | e.g. PASS - 5/5 replies |
| Ping endpoint A ke border router | e.g. PASS - 5/5 replies |
| Identitas/hash active dataset | e.g. SHA - 256 dari dataset export yang disimpan privat |
| Lokasi log attachment | e.g. C:\logs\cldt\endpoint - a - attach.txt |


## **Step 07: Mengubah “UDP Berhasil Sekali” Menjadi Gerbang Teknik**
**Track:** **`Cold Boot - Soak - Stop Rule`**

Repository mensyaratkan “repeatable cold boot” dan “sustained UDP”, tetapi tidak menetapkan jumlah boot, durasi soak, interval kirim, atau delivery threshold. Angka tersebut tidak boleh diisi seolah berasal dari repo. Mereka dibekukan setelah pilot singkat dan sebelum uji gate. Untuk smoke gate yang ketat, setiap reset tak terduga atau detach selama jendela ukur membuat repetisi itu gagal.

### **Parameter yang dibekukan sebelum gate**

| Parameter gate | Contoh awal - bekukan sebelum gate | Dasar pemilihan |
| - | - | - |
| Jumlah cold boot | e.g. 3 repetisi berurutan | jumlah yang muat waktu; 3 adalah smoke minimum dan 5 memberi bukti integrasi lebih kuat |
| Durasi tanpa daya | e.g. 10 s | cukup untuk membedakan cold boot dari warm reset |
| Batas waktu attachment | e.g. 60 s | di atas variasi pilot, lalu tetap untuk semua repetisi |
| Durasi UDP soak | e.g. 300 s | cukup untuk menghasilkan banyak datagram dan menangkap reset/detach |
| Interval pengiriman | e.g. 1 s | tidak membanjiri CLI/serial dan tetap memberi sample memadai |
| Port UDP | e.g. 1212 | satu port legal yang sama untuk seluruh repetisi |
| Delivery ratio minimum | e.g. 0.99 | dipilih sebelum melihat hasil gate |
| Reset tak terduga yang diizinkan | e.g. 0 | interpretasi gate yang ketat |
| Detach selama soak yang diizinkan | e.g. 0 | interpretasi gate yang ketat |

Jika interval pengiriman tetap, jumlah rencana datagram:

$$
\text{planned datagrams}
=
\left\lfloor
\frac{\text{soak duration}}{\text{send interval}}
\right\rfloor
$$

Delivery ratio dihitung dari sequence unik, bukan sekadar jumlah baris console:

$$
\text{delivery ratio}
=
\frac{\text{unique datagrams received}}{\text{datagrams sent}}
$$

Duplicate ratio dan gap tetap dilaporkan terpisah. Paket warm - up sebelum jendela soak tidak masuk pembilang atau penyebut.

### **Prosedur satu repetisi**

1. Power S3, RCP, dan endpoint dimatikan selama durasi tanpa daya yang sudah dibekukan; serial log sudah siap sebelum power kembali.
2. Ketiga board dinyalakan dengan urutan yang sama pada setiap repetisi. Urutan tersebut dicatat sekali dan tidak diubah untuk menyelamatkan run.
3. Waktu dari power - on sampai RCP siap, border router siap, dan endpoint attached dicatat.
4. Repetisi gagal bila attachment melewati batas waktu attachment yang sudah dibekukan, ada reset yang tidak direncanakan, atau endpoint detach pada jendela soak.
5. UDP memakai payload sequence yang meningkat. Waktu kirim dan terima dicatat pada log masing - masing sisi.
6. Setelah durasi soak berakhir, jumlah kirim, jumlah sequence unik yang diterima, duplicate, gap, reset, dan detach direkap.
7. Semua raw serial log dipertahankan. Repetisi gagal tidak dihapus dan tidak langsung diulang dengan parameter baru.
8. Perubahan wiring, firmware, dataset, channel, atau threshold mengakhiri block lama dan memulai block baru.

### **Lembar hasil**

| Repetisi | Attach time | Sent | Unique received | Duplicate | Gap | Reset | Detach | Status |
| - :| - :| - :| - :| - :| - :| - :| - :| - |
| 1 | e.g. 12.4 s | e.g. 300 | e.g. 300 | e.g. 0 | e.g. 0 | e.g. 0 | e.g. 0 | e.g. PASS |
| 2 | e.g. 11.9 s | e.g. 300 | e.g. 299 | e.g. 0 | e.g. 1 | e.g. 0 | e.g. 0 | e.g. FAIL |
| 3 | e.g. 12.1 s | e.g. 300 | e.g. 300 | e.g. 0 | e.g. 0 | e.g. 0 | e.g. 0 | e.g. PASS |
| 4 | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run |
| 5 | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run | e.g. not run |

| Ringkasan | Contoh format - ganti dengan hasil aktual |
| - | - |
| Repetisi valid | e.g. 2 |
| Repetisi gagal | e.g. 1 |
| Aggregate delivery ratio | e.g. 0.9989 |
| Total reset tak terduga | e.g. 0 |
| Total detach | e.g. 0 |
| Verdict gate | e.g. PASS atau FAIL |
| Alasan verdict | e.g. FAIL - repetition 2 memiliki satu sequence gap |



> **Stop rule repository:** bila jalur satu endpoint belum repeatable pada batas ke - 10, model dan control depth dipotong. Endpoint kedua, protocol CLDT, serta aktuasi tidak dipakai untuk “menutupi” kegagalan integrasi.


## **Urutan 1-10: Alokasi Kerja Tanpa Menambah Scope**

| No. | Fokus | Hasil yang dibawa ke langkah berikutnya |
| - :| - | - |
| 1 | inventaris checkout, label board, dan uji kabel satu per satu | mapping board - port yang tidak ambigu |
| 2 | verifikasi BOM, wiring resmi, powered hub, serta budget ceiling | tiga - board path siap dirakit; reserve tetap terlindungi |
| 3 | bekukan commit ESP - IDF, versi tool, byte contract, dan pin map | satu baseline konfigurasi yang dapat diulang |
| 4 | sediakan CMake/compiler lokal, build host scaffold, validasi delapan JSON | build boundary diketahui; skip tidak disalahartikan |
| 5 | isi `cldt_crc32c()` berdasarkan komentar TODO repo | implementasi portable pertama, belum diklaim benar |
| 6 | isi `test_crc32c.c`, jalankan fixed vectors, lalu ubah skip hanya bila lengkap | satu test repo benar - benar pass |
| 7 | build dan flash upstream `ot_rcp`; simpan `sdkconfig`, commit, log, dan hash | radio image yang identitasnya jelas |
| 8 | build `ot_br`, pasang UART S3 - RCP, dan tutup masalah Spinel | border router upstream stabil |
| 9 | build `ot_cli`, join `endpoint - a`, rekam role/IP, ping, dan UDP dua arah | jalur fisik minimum bekerja |
| 10 | jalankan cold - boot repetitions dan UDP soak dengan parameter yang sudah dibekukan | verdict gate beserta raw log dan alasan |

Pekerjaan bisa berjalan paralel - CRC pada host sambil menunggu hardware, misalnya - tetapi dependency tidak berubah. Spinel harus bekerja sebelum Thread dinilai; attachment harus stabil sebelum soak; gate harus lulus sebelum endpoint kedua atau project protocol ditambahkan.

Tidak ada file `.py` yang diimplementasikan pada fase ini. Delapan strict JSON dan delapan JSONC hanya melewati pemeriksaan syntax/schema dari workflow; notebook tidak mengisi manifest khusus Day 10 yang memang tidak ada di repo. `experiments/authoring/local - rtos - baseline.jsonc` menjadi lembar Week 2, sedangkan `experiments/authoring/baseline.jsonc` baru dibuka untuk baseline Week 3.



> **Akhir fase yang sah:** hasilnya boleh berupa PASS atau FAIL yang lengkap. FAIL dengan log, konfigurasi, dan diagnosis adalah bukti engineering. PASS tanpa hash, raw log, atau kriteria yang dibekukan sebelumnya bukan evidence yang cukup.
